In [1]:
import psycopg2
import pandas as pd
import seaborn as sns
import numpy as np
import pprint
from itertools import combinations
import pickle
import string
import re
import config.config as conf
from constants import CONSTANTS
import json

In [2]:
# lang='python'
lang='python'

In [3]:
conf

<module 'config.config' from '/home/mghan/sopjt/git/stackoverflow_src_2425/config/config.py'>

In [ ]:
conn = psycopg2.connect(dbname  =   conf.database_user['dbname'], 
                        user    =   conf.database_user['user'], 
                        password=   conf.database_user['password'], 
                        host    =   conf.database_user['host'], 
                        port    =   conf.database_user['port'])

In [ ]:
lang_tag_dict = {'python' : 'python',
                'cpp': 'c++',
                'java':'java',
                'vba':'vba'
                }

In [ ]:
for idx in range(len(CONSTANTS.monthly_timestamps)-1):
    st_dt = CONSTANTS.monthly_timestamps[idx].replace('.','-')
    end_dt = CONSTANTS.monthly_timestamps[idx+1].replace('.','-')

    cur = conn.cursor()
    sql = f"""select p.id, p.creationdate, p.title, p.tags, p2.body from public_for_2324.posts p , public_for_2324.postsbody p2 where p.id = p2.id and p.posttypeid = '1' and p.tags like '%<{lang}>%' and p.creationdate >=  '{str(st_dt)}'  and p.creationdate < '{str(end_dt)}' """
    print(sql)
    cur.execute(sql)
    inserted_data = cur.fetchall()
    type(inserted_data)
    cur.close()
    

    dict_q = [{ 'id' : x[0], 
                'creationdate' : x[1].isoformat(),
                'title' : x[2],
                'tags' : x[3],
                'body' : x[4]
            } for x in inserted_data]
    # json_str = json.dumps(dict_q, default=str, ensure_ascii=False, indent=2)

    with open(f"../../../so_pjt_src/data/snapshot2/questions/{lang}/{idx}.json", "w", encoding="utf-8") as f:
        json.dump(dict_q, f, ensure_ascii=False, indent=2)



In [ ]:
conn.close()